# EOM calibration: joint physical fit
This notebook uses the **same analysis as the main GUI**, with no instrument access.
Peter's and Brandon's notebooks are retained unchanged for comparison.

Fit both detector responses with one shared phase and separate amplitudes/backgrounds:
`theta = kA*VA + kB*VB + phi`; `R1 = C1 + A1*sin(theta)^2`; `R2 = C2 + A2*cos(theta)^2`.
All per-setting counts and recorded dwell times enter the Poisson fit; there is no training/held-out split.
Joint voltage selection targets `R1/(R1+R2)` over all eight cases, then minimizes the sum of six squared
voltages over equivalent phase allocations and periodic branches within the measured range and operating limits.

See [the full calibration guide](../pulsed_GUI/EOM_CALIBRATION_ANALYSIS.md) for assumptions, diagnostics and verification.


In [ ]:
from pathlib import Path
import sys, json
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p/'pulsed_GUI/eom_calibration.py').is_file()), None)
if ROOT is None: raise RuntimeError('Open this notebook within QKD_Code.')
sys.path.insert(0, str(ROOT/'pulsed_GUI'))
import eom_calibration as calibration
import recent_data
from IPython.display import Image, display
saved = recent_data.read(ROOT/'pulsed_GUI/gui_settings.json')
latest = recent_data.latest('sweep', saved, ROOT)
print('Latest completed sweep:', latest)


## Set the input and operating margins
Keep CSV/JSON/JSONL sidecars together. Review the selected path before running.
Limits are **EOM target volts**, not DAQ volts. ±180 V is a software margin, not a hardware rating.
The Vpi lower bound must exceed scan spacing. Acceptance tolerances are requirements, not noise weights.
Do not change them just to obtain a PASS. Legacy NI CSVs require an explicit approximate dwell.


In [ ]:
values = {key: default for key, label, default, kind in calibration.FORM}
values.update({
    'action': calibration.FIT,
    'csv': str(latest) if latest else '',  # Or enter a completed Detector_Traces.csv path.
    'alice-limit': '180', 'bob-limit': '180',
    'fit-vpi-min': '50', 'fit-vpi-max': '400',
    'balance-tolerance': '0.05', 'extinction-tolerance': '0.15',
    'plot-dpi': '300', 'legacy-dwell': '',
    'output': '',  # Default: source sweep / analysis / timestamp.
    'note': '',
})
print('Selected CSV:', values['csv'])


In [ ]:
output = calibration.analyze(calibration.settings(values))
report_path = output/'analysis.json'
report = json.loads(report_path.read_text(encoding='utf-8'))
for name in ('response_fit', 'scan_residuals', 'predicted_fractions', 'predicted_matrix', 'response_slices'):
    path = output/f'{name}.png'
    if path.exists(): display(Image(filename=str(path)))


## Copy calibration into the app
Copy the **whole JSON block** below, then use **Voltages → Paste calibration** in Polarization matrix or BB84.
The app also offers **Load latest calibration**. Imports fill all three profiles without starting instruments.
All fractions here are fitted predictions; a high R² does not verify the operating voltages.
No transfer is exported for unidentified fits, unmet fraction tolerances or infeasible voltage margins.
Residual warnings travel with eligible candidates. Physical H/R/V/L names remain provisional.


In [ ]:
transfer_path = output/'calibration_for_app.json'
if transfer_path.exists():
    print(transfer_path.read_text(encoding='utf-8'))
else:
    print('No accepted transfer:', report['rejection_reason'])


## Independent verification
Record all eight settings in **Polarization matrix** at the imported voltages.
Use **EOM calibration analysis → Verify fitted voltages** with the original analysis.json and that new run,
or the optional cell below. All exact voltage pairs and physical detector numbers are checked using ungated counts.
The voltages stay frozen. Exact conditional-binomial intervals receive a Bonferroni correction across eight checks;
all must lie inside the original tolerances to pass. Coverage assumes stationary independent detections,
not drift or voltage errors. The result applies to the saved verification conditions and does not certify absolute polarization labels.


In [ ]:
VERIFICATION_FOLDER = ''  # Completed polarization run folder; leave blank until measured.
if VERIFICATION_FOLDER:
    check_values = dict(values, **{'action': calibration.VERIFY,
        'fit-report': str(report_path), 'verification-folder': VERIFICATION_FOLDER,
        'confidence': '0.95', 'output': ''})
    verification_output = calibration.analyze(calibration.settings(check_values))
    display(Image(filename=str(verification_output/'verification.png')))
